# Scenario 15: Cost Tracking — Native SDK + Built-in OpenAI Judge

First of three cost-tracking demos split out from `14_cost_tracking.ipynb`.

**Focus:** native Galileo auto-instrumentation (`from galileo.openai import openai`) and the platform's **built-in** scorers (`GalileoMetrics.correctness`), which run on Galileo's default OpenAI-backed judge — no custom-LLM integration involved.

## What this notebook covers

| Q&A § | Topic | Step |
|---|---|---|
| 1.2 | Pricing config (read-only GET) | Step 3 |
| 1.5 | Cost formula vs API cost | Step 4 + 6 |
| 2.2 | `cost` vs `*_metric_cost` (built-in judge spend) | Step 5–6 |
| 1.4 | Cost APIs | Step 7 |
| 2.3 | Job progress | Step 8 |
| 2.1 | Sampling gap | Step 9 |
| 1.1 | Dashboard surfaces | Step 10 |

For `custom_LLM_boolean_metric` against a custom-integrated LLM, see `16_cost_tracking_custom_judge.ipynb`. For pure-OTel cost tracking with custom metrics, see `17_cost_tracking_otel.ipynb`.

## Note on `Num Reasoning Tokens` (§1.7)

This notebook uses `gpt-4o-mini`, a non-reasoning model. OpenAI does not emit `usage.completion_tokens_details.reasoning_tokens` for it, so the SDK extractor at `galileo-python/.../openai/extractors.py:648` writes `0` to `span.metrics.num_reasoning_tokens` — that's the **correct** behavior, not a bug. The trace's `Num Reasoning Tokens = 0` reading proves the pipeline is wired correctly; if you swap `MODEL` to an o-series model (`o1-mini`, `o3-mini`, `o4-mini`, `gpt-5-*`), real reasoning tokens will surface.

For demonstrations that *do* populate `num_reasoning_tokens` non-zero with a non-reasoning model:
- Notebook 16 monkey-patches `_parse_usage` to inject a synthetic value (native SDK path).
- Notebook 17 sets `gen_ai.usage.details.reasoning_tokens` directly on the OTel span (OTel ingest path).

## Step 0: Load environment variables

Requires `GALILEO_API_KEY` and `OPENAI_API_KEY` in `.env`.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

GALILEO_SDK_SOURCE = None
for root in [Path.cwd(), *Path.cwd().parents]:
    local_sdk_src = root / 'galileo-python' / 'src'
    if (local_sdk_src / 'galileo').exists():
        GALILEO_SDK_SOURCE = local_sdk_src
        if str(local_sdk_src) not in sys.path:
            sys.path.insert(0, str(local_sdk_src))
        break

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

for candidate in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

Loaded credentials from /Users/binliu/Documents/rungalileo/demos/galileo-test/.env


## Step 0b: Imports and constants

In [2]:
import json
import time
from urllib.parse import urljoin, quote

from galileo import GalileoMetrics, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.log_streams import create_log_stream, enable_metrics, get_log_stream
from galileo.openai import openai
from galileo.projects import delete_project
from galileo.resources.api.trace import query_traces_projects_project_id_traces_search_post
from galileo.resources.models.log_records_query_request import LogRecordsQueryRequest
from galileo_core.constants.request_method import RequestMethod

PROJECT_NAME = 'GalileoEval_S15_CostNative'
LOG_STREAM_NAME = 'cost-native'
MODEL = 'gpt-4o-mini'

print(f'Galileo SDK source: {GALILEO_SDK_SOURCE or "installed package"}')
PROJECT_NAME, LOG_STREAM_NAME, MODEL

/Users/binliu/Documents/rungalileo/demos/galileo-test/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Galileo SDK source: /Users/binliu/Documents/rungalileo/galileo-python/src


('GalileoEval_S15_CostNative', 'cost-native', 'gpt-4o-mini')

## Step 1: Initialize Galileo and capture URLs

In [3]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

ls = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME) or \
     create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger_inst = galileo_context.get_logger_instance()
project_id = getattr(logger_inst, 'project_id', None)
log_stream_id = getattr(logger_inst, 'log_stream_id', None)

console_base = str(config.console_url)
project_url = urljoin(console_base, f'project/{project_id}')
log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
trends_url = f'{log_stream_url}?tab=trends'

print('Log stream :', log_stream_url)
print('Trends tab :', trends_url)

Log stream : https://console-bin-citizens.gcp-dev.galileo.ai/project/2b351cf9-73db-4c77-a6d7-18c242b54214/log-streams/63c60b04-0561-4d37-a77b-4b3b0b107568
Trends tab : https://console-bin-citizens.gcp-dev.galileo.ai/project/2b351cf9-73db-4c77-a6d7-18c242b54214/log-streams/63c60b04-0561-4d37-a77b-4b3b0b107568?tab=trends


## Step 2: Wrapped OpenAI client (auto-instrumented)

`from galileo.openai import openai` is a drop-in replacement for the stock SDK. Every `chat.completions.create()` call emits one Galileo trace, and the runners' Celery cost job prices each LLM step using the captured token counts and `params.model` (`runners/runners/utils/costs/cost.py:13-107`).

In [4]:
client = openai.OpenAI()
client

## Step 3 — §1.2: Pricing config — read the default for `gpt-4o-mini`

The pricing tier hierarchy (read-only here):

1. **Per-org override** — `model_price_overrides` table (`api/api/routers/model_price.py:89`)
2. **Static fallback** — `orbit/.../costs/data/costs.json`
3. **Unknown model** — `cost = None` (`runners/.../cost.py:96`)

We just call the GET endpoint to see the default + any override that already applies in this env. Notebook 16 demonstrates upsert/delete; this notebook stays read-only so the pricing surface stays intact for the rest of the run.

In [5]:
prices = config.api_client.request(
    method=RequestMethod.GET,
    path=f'/models/{quote(MODEL, safe="")}/prices',
)
print(f'--- Prices for {MODEL} ---')
print(json.dumps(prices, indent=2, default=str))

--- Prices for gpt-4o-mini ---
{
  "default": {
    "input_price": 0.15,
    "output_price": 0.6
  }
}


## Step 4 — §1.5: Send native traces, capture token counts

We send 6 single-turn prompts. The wrapped client auto-emits one trace per call. Token counts come from `response.usage`.

**SDK note:** newer `openai` versions expose `usage.input_tokens`/`output_tokens`; older ones used `prompt_tokens`/`completion_tokens`. The `_usage_in_out` shim accepts either.

In [6]:
def _usage_in_out(usage):
    """Return (input_tokens, output_tokens) accepting either openai SDK naming."""
    if usage is None:
        return 0, 0
    in_t = getattr(usage, 'input_tokens', None) or getattr(usage, 'prompt_tokens', None) or 0
    out_t = getattr(usage, 'output_tokens', None) or getattr(usage, 'completion_tokens', None) or 0
    return in_t, out_t

prompts = [
    'How do I reset my password?',
    'What are your business hours?',
    'Can I get a refund on my last order?',
    'Where is my package right now?',
    'How do I contact a human agent?',
    'Is there a mobile app I can download?',
]

galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

native_usage = []
for prompt in prompts:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'You are a concise customer support agent. Answer in 1-2 sentences.'},
            {'role': 'user', 'content': prompt},
        ],
        max_tokens=80,
    )
    in_t, out_t = _usage_in_out(response.usage)
    native_usage.append({'prompt': prompt, 'in_tokens': in_t, 'out_tokens': out_t})

galileo_context.flush()

for row in native_usage:
    print(f"{row['in_tokens']:>4} in / {row['out_tokens']:>4} out  |  {row['prompt']}")
print(f'\n-> {len(native_usage)} native traces sent.')

  34 in /   29 out  |  How do I reset my password?
  33 in /   16 out  |  What are your business hours?
  37 in /   18 out  |  Can I get a refund on my last order?
  34 in /   26 out  |  Where is my package right now?
  35 in /   22 out  |  How do I contact a human agent?
  36 in /   31 out  |  Is there a mobile app I can download?

-> 6 native traces sent.


## Step 5 — §2.2: Enable Galileo's built-in `correctness` metric

`correctness` is a built-in scorer. Galileo runs it using its default judge configuration (an OpenAI-backed model managed by the platform — no custom integration required from you).

Every time the judge fires, its own LLM spend lands in the **`correctness_metric_cost`** column on the metrics table. That's the §2.2 "two cost streams" demonstration in this notebook:

| Column | What it represents |
|---|---|
| `cost` | User-trace LLM cost (your gpt-4o-mini calls in Step 4) |
| `correctness_metric_cost` | Galileo's built-in correctness judge spend |

`enable_metrics()` replaces the full active set on this log stream — list everything you want active here.

In [7]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[GalileoMetrics.correctness],
)
print(f'Metrics enabled on {LOG_STREAM_NAME}: [correctness]')

Metrics enabled on cost-native: [correctness]


## Step 6 — §1.5 + §2.2: Fetch metrics, verify cost formula, show both streams

1. **Verify the cost formula** — compute `expected_cost = (in_tokens × in_price + out_tokens × out_price) / 1e6` from `costs.json`, compare with the API-returned `cost`. A small positive diff is the runners' `formatting_tokens × message_count + response_prefix_tokens` adjustment (`orbit/libs/python/promptgalileo/promptgalileo/schemas/model.py:155-207`).
2. **Show both streams** — print `cost` and `correctness_metric_cost` side by side.

We poll up to 90s for the runners + scorer Celery jobs to populate the columns. If `cost` stays `None`, see Step 8 to inspect job state.

**Where each value comes from in the trace-search response** (relevant for §2.2):
- `record.metric_info.additional_properties["cost"].value` → the trace's user `cost` (single number).
- `record.metric_info.additional_properties[scorer_name].cost` → that scorer's own `metric_cost`. Note this is `MetricSuccess.cost` (a typed attribute on the schema, see `galileo-python/src/galileo/resources/models/metric_success.py:122`), **not** a key inside `additional_properties`.

A small but important nuance: `MetricSuccess.cost` is `Union[None, Unset, float]`. We can't tell "judge didn't run yet" from "judge ran but reported no cost" by reading this field alone — we have to check `status_type` too. The diagnostic below distinguishes three states: *populated with a number*, *explicitly null*, and *not yet emitted by the scorer at all*.

In [8]:
GPT_4O_MINI_IN_PER_M = 0.15
GPT_4O_MINI_OUT_PER_M = 0.60
SCORER_NAME = 'correctness'

def fetch_traces(target_log_stream_id, limit=50):
    req = LogRecordsQueryRequest(log_stream_id=target_log_stream_id, limit=limit)
    return query_traces_projects_project_id_traces_search_post.sync(
        project_id=project_id, client=config.api_client, body=req,
    )

def _props(obj, attr):
    target = getattr(obj, attr, None)
    if target is None or not hasattr(target, 'additional_properties'):
        return {}
    return target.additional_properties or {}

def extract_user_cost(record):
    info = _props(record, 'metric_info')
    cost_metric = info.get('cost')
    return getattr(cost_metric, 'value', None) if cost_metric else None

def extract_scorer_metric_cost(record, scorer_name):
    """Return (state, value).
    state in {'populated', 'null', 'absent'}: populated=numeric, null=judge ran but cost is None,
    absent=scorer key missing entirely (judge hasn't run yet).
    """
    info = _props(record, 'metric_info')
    entry = info.get(scorer_name)
    if entry is None:
        return 'absent', None
    cost = getattr(entry, 'cost', None)
    if isinstance(cost, (int, float)):
        return 'populated', float(cost)
    return 'null', None

def extract_tokens(record):
    """Token counts live as typed fields on LlmMetrics (not additional_properties)."""
    metrics_obj = getattr(record, 'metrics', None)
    if metrics_obj is None:
        return 0, 0
    in_t = getattr(metrics_obj, 'num_input_tokens', None) or 0
    out_t = getattr(metrics_obj, 'num_output_tokens', None) or 0
    if in_t == 0 and out_t == 0:
        ap = _props(record, 'metrics')
        in_t = ap.get('num_input_tokens') or in_t
        out_t = ap.get('num_output_tokens') or out_t
    return in_t, out_t

deadline = time.time() + 90
records = []
while time.time() < deadline:
    resp = fetch_traces(log_stream_id)
    records = getattr(resp, 'records', []) or []
    if records and any(extract_user_cost(r) is not None for r in records):
        break
    time.sleep(5)

print(f'{"in/out":>10}  {"api cost":>14}  {"expected":>14}  {"diff":>10}  {"metric_cost":>14}  prompt')
print('-' * 110)
total_user = 0.0
metric_cost_states = {'populated': 0, 'null': 0, 'absent': 0}
total_metric = 0.0
for r in records[:10]:
    in_t, out_t = extract_tokens(r)
    cost = extract_user_cost(r)
    state, metric_cost = extract_scorer_metric_cost(r, SCORER_NAME)
    metric_cost_states[state] += 1
    expected = (in_t * GPT_4O_MINI_IN_PER_M + out_t * GPT_4O_MINI_OUT_PER_M) / 1_000_000
    diff = (cost - expected) if isinstance(cost, (int, float)) else None
    raw = getattr(r, 'input_', '') or ''
    prompt = (raw if isinstance(raw, str) else '')[:40]
    if isinstance(cost, (int, float)): total_user += cost
    if state == 'populated': total_metric += metric_cost
    metric_cost_disp = f'${metric_cost:.6f}' if state == 'populated' else state
    print(
        f'{in_t:>4}/{out_t:>4}'
        f'  {(f"${cost:.6f}" if isinstance(cost,(int,float)) else "None"):>14}'
        f'  {f"${expected:.6f}":>14}'
        f'  {(f"+${diff:.6f}" if diff is not None else "n/a"):>10}'
        f'  {metric_cost_disp:>14}'
        f'  {prompt}'
    )

print(f'\nSum cost                       = ${total_user:.6f}')
print(f'Sum {SCORER_NAME}_metric_cost  = ${total_metric:.6f}'
      f'  (populated={metric_cost_states["populated"]},'
      f' null={metric_cost_states["null"]},'
      f' absent={metric_cost_states["absent"]})')

if metric_cost_states['populated'] == 0:
    print(
        f'\n[Note] {SCORER_NAME} is a built-in Galileo-managed judge. Its per-trace metric_cost '
        f'often surfaces as `null` (judge ran but cost not attributed at this layer) rather than '
        f'a number — that is expected for built-ins. Notebook 16 demonstrates the populated case '
        f'with a custom-integrated judge model.'
    )

    in/out        api cost        expected        diff     metric_cost  prompt
--------------------------------------------------------------------------------------------------------------
36.0/31.0       $0.000024       $0.000024  +$0.000000          absent  {"messages": [{"content": "You are a con
35.0/22.0       $0.000018       $0.000018  +$0.000000          absent  {"messages": [{"content": "You are a con
34.0/26.0       $0.000021       $0.000021  +$-0.000000          absent  {"messages": [{"content": "You are a con
37.0/18.0       $0.000016       $0.000016  +$0.000000          absent  {"messages": [{"content": "You are a con
33.0/16.0       $0.000015       $0.000015  +$-0.000000          absent  {"messages": [{"content": "You are a con
34.0/29.0       $0.000023       $0.000023  +$0.000000          absent  {"messages": [{"content": "You are a con

Sum cost                       = $0.000117
Sum correctness_metric_cost  = $0.000000  (populated=0, null=0, absent=6)

[Note] correctnes

## Step 7 — §1.4: Cost APIs (live calls)

| Endpoint | Returns |
|---|---|
| `GET /projects/{pid}/log_streams/{lsid}/trends` | Trends dashboard config |
| `POST /projects/{pid}/log_streams/{lsid}/logstream_insights/token_usage` | Per-record token usage |
| `POST /projects/{pid}/metrics/search` | Per-record metrics (already exercised in Step 6) |

In [9]:
trends_path = f'/projects/{project_id}/log_streams/{log_stream_id}/trends'
trends_resp = config.api_client.request(method=RequestMethod.GET, path=trends_path)
sections = (trends_resp or {}).get('sections') or []
metric_keys = set()
for s in sections:
    for w in (s.get('widgets') or []):
        metric_keys.update((w.get('metric_keys') or []))
        if w.get('metric_key'):
            metric_keys.add(w['metric_key'])
cost_metrics = sorted(k for k in metric_keys if 'cost' in k.lower())
print('--- Trends ---')
print(f'  sections: {len(sections)}')
print(f'  cost-related keys: {cost_metrics or "(none preconfigured)"}')

token_path = f'/projects/{project_id}/log_streams/{log_stream_id}/logstream_insights/token_usage'
tok_resp = config.api_client.request(
    method=RequestMethod.POST, path=token_path,
    json={'limit': 50, 'starting_token': 0},
)
rows = tok_resp.get('records') or tok_resp.get('items') or []
print(f'\n--- Token usage ({len(rows)} records) ---')
print(f'  sum input_tokens  = {sum((r.get("input_tokens") or 0) for r in rows)}')
print(f'  sum output_tokens = {sum((r.get("output_tokens") or 0) for r in rows)}')

print(f'\n--- Metrics search (Step 6 totals) ---')
print(f'  total cost                  = ${total_user:.6f}')
print(f'  total correctness_metric_cost = ${total_metric:.6f}')

--- Trends ---
  sections: 1
  cost-related keys: (none preconfigured)



--- Token usage (0 records) ---
  sum input_tokens  = 0
  sum output_tokens = 0

--- Metrics search (Step 6 totals) ---
  total cost                  = $0.000117
  total correctness_metric_cost = $0.000000


## Step 8 — §2.3: Job progress

`Job` table (`api/api/models.py:1140, 1147, 1156-1158`). For a log stream, `run_id` *is* `log_stream_id`. If `comet` (Celery Beat) isn't running, `requeue_metrics_trigger` never fires and stuck jobs stay stuck.

In [10]:
jobs = config.api_client.request(
    method=RequestMethod.GET,
    path=f'/projects/{project_id}/runs/{log_stream_id}/jobs',
) or []

print(f'Total jobs for this log stream: {len(jobs)}\n')
print(f'{"job_name":<28} {"status":<14} {"progress":>10}  message')
print('-' * 80)
for j in jobs[:15]:
    name = (j.get('job_name') or '')[:28]
    status = (j.get('status') or '')[:14]
    sc, st = j.get('steps_completed'), j.get('steps_total')
    pct = f'{(sc/st*100):.0f}%' if (sc is not None and st) else '-'
    msg = (j.get('progress_message') or '')[:40]
    print(f'{name:<28} {status:<14} {pct:>10}  {msg}')

Total jobs for this log stream: 4

job_name                     status           progress  message
--------------------------------------------------------------------------------
log_stream_scorer            completed               -  
log_stream_scorer            completed               -  
log_stream_scorer            completed               -  
log_stream_scorer            completed               -  


## Step 9 — §2.1: Sampling — the honest answer

Sampling is a per-scorer property on the **job request** (`SegmentFilter.sample_rate`, `orbit/.../schemas/filter.py:129`). The public `enable_metrics()` doesn't expose it — confirmed by inspecting the signature.

In [11]:
import inspect
sig = inspect.signature(enable_metrics)
for p in sig.parameters.values():
    print(f'  - {p}')
assert 'sample_rate' not in sig.parameters
print('\nConfirmed: SDK does not expose sample_rate on enable_metrics().')

  - log_stream_name: str | None = None
  - project_name: str | None = None
  - metrics: list[galileo.schema.metrics.GalileoMetrics | galileo.schema.metrics.Metric | galileo.schema.metrics.LocalMetricConfig | str]

Confirmed: SDK does not expose sample_rate on enable_metrics().


## Step 10 — §1.1: Dashboard surfaces

In [12]:
print('=== UI surfaces ===')
print(f'Log stream       : {log_stream_url}')
print(f'Trends tab       : {trends_url}')
print(f'Settings/pricing : {urljoin(console_base, "settings/model-pricing")}')

=== UI surfaces ===
Log stream       : https://console-bin-citizens.gcp-dev.galileo.ai/project/2b351cf9-73db-4c77-a6d7-18c242b54214/log-streams/63c60b04-0561-4d37-a77b-4b3b0b107568
Trends tab       : https://console-bin-citizens.gcp-dev.galileo.ai/project/2b351cf9-73db-4c77-a6d7-18c242b54214/log-streams/63c60b04-0561-4d37-a77b-4b3b0b107568?tab=trends
Settings/pricing : https://console-bin-citizens.gcp-dev.galileo.ai/settings/model-pricing


## Step 11: Cleanup

Uncomment `delete_project(...)` to tear down. Comment it out to keep the data for further inspection.

In [13]:
# delete_project(name=PROJECT_NAME)
print(f'Project kept for inspection: {PROJECT_NAME}')
print(f'  Log stream : {log_stream_url}')

Project kept for inspection: GalileoEval_S15_CostNative
  Log stream : https://console-bin-citizens.gcp-dev.galileo.ai/project/2b351cf9-73db-4c77-a6d7-18c242b54214/log-streams/63c60b04-0561-4d37-a77b-4b3b0b107568
